<h1><center>Computational Quantum Physics - PHYS 463</center></h1>

<p><center> <b>Lecturer:</b> <i>Prof. G. Carleo</i> </center><p>
    
<p><center> <b>Assistants: </b> <i>gabriel.pescia@epfl.ch, alessandro.sinibaldi@epfl.ch, clemens.giuliani@epfl.ch, linda.mauron@epfl.ch </i> </center><p>

# Exercise 8 - Template

In [ ]:
# Run this if you are on Windows and you don't want to use GPU
import os

os.environ["JAX_PLATFORM_NAME"] = "cpu"

In [ ]:
from typing import Any, Tuple
from flax import nnx
import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import netket as nk
import numpy as np

In [ ]:
# Setup

# Number of spins
N = 20
# Number of samples
Ns = 1024
# How many Markov chains to run in parallel
n_chains = 16
# How many samples to discard
n_discard = 128

# Graph
g = nk.graph.Chain(N, pbc=True)
# Hilbert space
hi = nk.hilbert.Spin(s=1 / 2, N=g.n_nodes)
# Hamiltonian
ha = nk.operator.Ising(hilbert=hi, graph=g, h=1)
# Sampler (single-spin flips)
sa = nk.sampler.MetropolisLocal(hi, n_chains=n_chains)
# Optimizer
op = nk.optimizer.Sgd(0.01)

In [ ]:
# Exact diagonalization
E0 = nk.exact.lanczos_ed(ha)

In [ ]:
class FFNN(nn.Module):
    # Number of hidden neurons per hidden layer
    width: int = 5
    # Number of hidden layers
    n_layers: int = 1

    @nn.compact
    def __call__(self, x):
        # x has shape (n_samples, N)
        # Hidden layers: x^(l) = ReLU(W^(l-1) x^(l-1) + b^(l))
        for _ in range(self.n_layers):
            x = nn.relu(nn.Dense(self.width)(x))
        # Output layer with a single unit: log psi(s) = ReLU(W x + b)
        x = nn.relu(nn.Dense(1)(x))
        # (n_samples, 1) -> (n_samples,)
        return x.squeeze(-1)

In [ ]:
ma = FFNN(width=N // 4, n_layers=1)

In [ ]:
# Variational state object
vs = nk.vqs.MCState(sa, ma, n_samples=Ns, n_discard_per_chain=n_discard)
# VMC optimization driver
vmc = nk.VMC(ha, op, variational_state=vs)

In [ ]:
vmc.run(n_iter=1000)

In [ ]:
ffnn_energy = vs.expect(ha)
error = abs((ffnn_energy.mean - E0) / E0)
print("Optimized energy and relative error: ", ffnn_energy, error)

In [ ]:
# ...